# Part 1 — DNS Tunneling Feature Extraction (PCAP → tabular features)

This section turns raw PCAP captures from [DNS-Tunnel-Datasets](https://github.com/ggyggy666/DNS-Tunnel-Datasets) into the kind of feature table the rest of this notebook already expects (`train.csv` / `val.csv` / `test.csv` with a `Label` column) — replacing the original NSL-KDD-style tabular intrusion data with DNS-specific features aimed at spotting DNS tunneling. The label scheme is kept as generic `benign` / `tunnel` so it's easy to extend to other DNS-layer attacks later.

Dataset contents: benign DNS traffic from the top 1M Cloudflare domains (`normal/`), traffic from six DNS tunneling tools with several supported query-record types each (`tunnel/`), traffic from unseen tunneling tools for robustness testing (`unkownTunnel/`), an Android-platform tunnel client (`crossEndPoint/`), and wildcard DNS traffic that's benign but structurally resembles tunneling (`wildcard/`). See the [dataset paper](https://ieeexplore.ieee.org/document/10636232) for details.

In [ ]:
import importlib.util
import os
import subprocess
import sys
import time

import pandas as pd
import matplotlib.pyplot as plt

if importlib.util.find_spec("dpkt") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "dpkt"])
import dpkt
print("dpkt ready:", getattr(dpkt, "__version__", "ok"))

dpkt ready: 1.9.8


In [2]:
DATASET_REPO = "https://github.com/ggyggy666/DNS-Tunnel-Datasets.git"
DATA_DIR = "./DNS-Tunnel-Datasets"
CACHE_FILES = ['dataset/train.csv', 'dataset/val.csv', 'dataset/test.csv', 'dataset/heldout.csv']

if all(os.path.exists(f) for f in CACHE_FILES):
    print("Found existing extracted feature CSVs in dataset/ -- skipping dataset clone")
    print("(the 845MB raw PCAP dataset isn't needed to reuse already-extracted features).")
elif not os.path.isdir(DATA_DIR):
    print(f"Cloning {DATASET_REPO} into {DATA_DIR} ... (~845MB total, only needed once)")
    subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, DATA_DIR], check=True)
else:
    print(f"Found existing dataset at {DATA_DIR}, skipping clone.")

Found existing extracted feature CSVs in dataset/ -- skipping dataset clone
(the 845MB raw PCAP dataset isn't needed to reuse already-extracted features).


In [3]:
# dns_feature_extraction.py must sit next to this notebook -- it's the
# reusable PCAP-parsing / feature-engineering module built for this project.
from dns_feature_extraction import build_dataset, to_ml_frame

## Choosing files: in-distribution train/val/test + a held-out robustness set

Every capture in this dataset is one whole session for one traffic type, so the split is done **by file**, not by row (splitting rows would leak the same tunnel session across train and test). A small, diverse subset is used for this first pass — enough to validate the pipeline in about a minute and a half; see the caveats section at the end for how to scale to all 131 files.

* **Train** — 2 benign captures + 4 tunnel captures spanning 4 different tools/record types (dnscat2-cname, dnscat2-txt, iodine-txt, DNS-shell).
* **Val** — 1 benign capture + 1 tunnel capture (`tuns`, a tool not seen in train) — used for early stopping / model selection.
* **Test** — 1 benign capture + 1 tunnel capture (`dnspot`, also not seen in train) — the usual held-out accuracy number.
* **Heldout (robustness)** — never trained or tuned on: two tunneling tools the model has never seen (`unkownTunnel`), an Android tunnel client (`crossEndPoint`), and benign wildcard DNS traffic that looks structurally like tunneling (`wildcard`). This is where you'd check whether the model generalizes to *unseen* attack tools, and whether it false-positives on legitimate high-subdomain-count traffic.

In [ ]:
DATA_DIR = "./DNS-Tunnel-Datasets"

# ---- in-distribution files: used to fit and select the model ----------
TRAIN_FILES = [
    (f"{DATA_DIR}/normal/normal/normal_00000_20230805150331.pcap", "benign"),
    (f"{DATA_DIR}/normal/normal/normal_00001_20230805152732.pcap", "benign"),
    (f"{DATA_DIR}/tunnel/dnscat2-cname.pcap", "tunnel"),
    (f"{DATA_DIR}/tunnel/dnscat2-txt.pcap", "tunnel"),
    (f"{DATA_DIR}/tunnel/iodine-txt.pcap", "tunnel"),
    (f"{DATA_DIR}/tunnel/DNS-shell.pcap", "tunnel"),
    # Hard negatives: wildcard DNS is benign but structurally tunnel-like
    # (long, high-entropy subdomains). Without any wildcard-shaped benign
    # example in training, the model only ever saw "benign = short,
    # low-entropy qname" and learned qname length/entropy as a shortcut
    # for "tunnel" -- which is exactly why it flagged 89% of held-out
    # wildcard traffic as tunneling. These two files teach it that
    # long/high-entropy alone isn't enough; the wildcard file already in
    # HELDOUT_FILES (00000) is a *different* pcap, so this is still an
    # unseen-session generalization test, not leakage.
    (f"{DATA_DIR}/wildcard/wildcard_00001_20231011215715.pcap", "benign"),
    (f"{DATA_DIR}/wildcard/wildcard_00002_20231011220330.pcap", "benign"),
    # Hard positive: dnspot is the one tunnel tool in this dataset that
    # doesn't give itself away with a short response TTL or TXT records --
    # its response_min_ttl averages ~3586s (indistinguishable from ordinary
    # long-TTL benign traffic) versus ~34s for the tunnel tools already in
    # TRAIN_FILES and ~4.6s for VAL's tuns. Without an example like that,
    # the model only ever sees "tunnel = short TTL" and a random init has
    # about 1-in-4 odds of misclassifying held-out dnspot traffic as benign.
    # dnspot_00001 is a different capture/session than TEST_FILES'
    # dnspot_00000, so this is still an unseen-session generalization test
    # for that file, not leakage.
    (f"{DATA_DIR}/tunnel/dnspot/dnspot_00001_20230909170527.pcap", "tunnel"),
]
VAL_FILES = [
    (f"{DATA_DIR}/normal/normal/normal_00002_20230805155315.pcap", "benign"),
    (f"{DATA_DIR}/tunnel/tuns.pcap", "tunnel"),
    (f"{DATA_DIR}/wildcard/wildcard_00003_20231011220812.pcap", "benign"),
]
TEST_FILES = [
    (f"{DATA_DIR}/normal/normal/normal_00003_20230805162300.pcap", "benign"),
    (f"{DATA_DIR}/tunnel/dnspot/dnspot_00000_20230909155901.pcap", "tunnel"),
]

# Held-out robustness set: unseen tools + unseen platform + a hard-negative
# benign capture. Deliberately NOT used for training/validation -- see the
# caveats section at the end of this notebook for how it's meant to be used.
# wildcard_00000 stays here (untouched) so before/after wildcard accuracy on
# THIS file is directly comparable to the original run.
HELDOUT_FILES = [
    (f"{DATA_DIR}/unkownTunnel/cobalstrike.pcap", "tunnel", "unkownTunnel"),
    (f"{DATA_DIR}/unkownTunnel/ozymandns.pcap", "tunnel", "unkownTunnel"),
    (f"{DATA_DIR}/crossEndPoint/AndIodine-TXT.pcap", "tunnel", "crossEndPoint"),
    (f"{DATA_DIR}/wildcard/wildcard_00000_20231011215521.pcap", "benign", "wildcard"),
]


In [ ]:
CACHE_FILES = ['dataset/train.csv', 'dataset/val.csv', 'dataset/test.csv', 'dataset/heldout.csv']

if all(os.path.exists(f) for f in CACHE_FILES):
    print("Found existing extracted CSVs -- loading them instead of re-parsing PCAPs.")
    print("Delete the files in dataset/ if you want to force a fresh extraction (e.g. after\n"
          "          changing TRAIN_FILES/VAL_FILES/TEST_FILES/HELDOUT_FILES above).")
    train_raw = pd.read_csv('dataset/train.csv')
    val_raw = pd.read_csv('dataset/val.csv')
    test_raw = pd.read_csv('dataset/test.csv')
    heldout_raw = pd.read_csv('dataset/heldout.csv')
else:
    t0 = time.time()
    print("=== TRAIN ===")
    train_raw = build_dataset(TRAIN_FILES)
    print("\n=== VAL ===")
    val_raw = build_dataset(VAL_FILES)
    print("\n=== TEST ===")
    test_raw = build_dataset(TEST_FILES)

    print("\n=== HELDOUT (robustness) ===")
    heldout_raw = build_dataset([(p, l) for p, l, _ in HELDOUT_FILES])
    heldout_raw["source_group"] = heldout_raw["pcap_file"].map({p: g for p, _, g in HELDOUT_FILES})

    print(f"\nExtraction finished in {time.time() - t0:.1f}s")

for name, d in [("train", train_raw), ("val", val_raw), ("test", test_raw), ("heldout", heldout_raw)]:
    print(f"{name:8s} rows={len(d):6d}  label_counts={d['Label'].value_counts().to_dict()}")

### Quick sanity check: do the extracted features actually separate the classes?

In [ ]:
summary_cols = ['qname_len', 'entropy', 'digit_ratio', 'hex_ratio',
                'domain_query_count', 'domain_unique_subdomain_ratio',
                'domain_query_rate', 'response_min_ttl']
print(train_raw.groupby('Label')[summary_cols].mean().T)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for label, color in zip(['benign', 'tunnel'], ['#4374B3', '#FF0B04']):
    subset = train_raw[train_raw['Label'] == label]
    axes[0].hist(subset['qname_len'], bins=40, alpha=0.6, label=label, color=color)
    axes[1].hist(subset['entropy'], bins=40, alpha=0.6, label=label, color=color)
axes[0].set_title('Query name length'); axes[0].set_xlabel('characters'); axes[0].legend()
axes[1].set_title('Query name entropy'); axes[1].set_xlabel('bits/char'); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('dataset', exist_ok=True)

train_ml = to_ml_frame(train_raw)
val_ml = to_ml_frame(val_raw)
test_ml = to_ml_frame(test_raw)
heldout_ml = to_ml_frame(heldout_raw, extra_cols=['source_group'])

train_ml.to_csv('dataset/train.csv', index=False)
val_ml.to_csv('dataset/val.csv', index=False)
test_ml.to_csv('dataset/test.csv', index=False)
heldout_ml.to_csv('dataset/heldout.csv', index=False)

print("Saved:")
for name, d in [('train', train_ml), ('val', val_ml), ('test', test_ml), ('heldout', heldout_ml)]:
    print(f"  dataset/{name}.csv  shape={d.shape}")

### Notes, caveats, and how to scale this up

**Why a hand-rolled DNS parser instead of `dpkt.dns.DNS`.** `dpkt`'s DNS decoder treats every label as strict UTF-8 and throws `UnicodeDecodeError` on anything else. DNS tunneling tools exist specifically to smuggle arbitrary bytes through DNS names and record data, so this isn't an edge case — on `iodine-txt.pcap` it silently dropped **97%** of DNS messages, and **46%** on the Android `AndIodine-TXT` capture. `dns_feature_extraction.py` parses the header and name fields by hand (`struct` + latin-1, a lossless 1-byte↔1-char mapping) so raw/random payload bytes never abort parsing. Worth remembering if you extend the parser: anything that assumes "DNS name = printable ASCII" will quietly bias a tunnel-detection dataset toward the *easy* tunnels.

**Why per-domain aggregate features, not just per-query ones.** A single DNS query for a high-entropy subdomain is only mildly suspicious — plenty of CDNs and trackers do that. What actually distinguishes a tunnel is the *pattern* across many queries to the same domain in a short window: hundreds/thousands of queries, almost all unique, arriving fast. That's what `domain_query_count`, `domain_unique_subdomain_ratio`, and `domain_query_rate` capture. `domain_query_rate` floors the time window at 1 second on purpose — without it, a domain seen only once or twice gets a division-by-near-zero "rate" that dwarfs genuinely bursty tunnel traffic.

**Why the train/val/test split is by whole file, not by row.** All queries in one capture come from the same session — splitting at the row level would let queries from the same tunnel session leak across train and test, inflating accuracy. Splitting whole files instead means test performance actually reflects generalization to a new session.

**Scaling to the full dataset (845 MB, 131 files).** This pass uses a small, hand-picked subset (~10% of the data, chosen to run in under two minutes) to validate the pipeline end-to-end. To use everything: glob every `*.pcap` under `normal/` and `tunnel/` and split file-wise with `sklearn.model_selection.GroupShuffleSplit` (stratified by label) instead of hand-picking files. Note a few tools have *both* a merged top-level file (e.g. `tunnel/dnspot.pcap`) *and* the same traffic split into chunks in a same-named subfolder — use one or the other, not both, or you'll double-count. Extraction throughput here was roughly 1.3 MB/s, so the full corpus is on the order of 10+ minutes single-threaded — parallelizing `extract_query_records` per file (`multiprocessing.Pool`) is the natural speed-up.

**`heldout.csv` is deliberately not wired into the model cells below.** It holds DNS tunneling tools the model never trains on (`unkownTunnel`: CobaltStrike, ozymandns; `crossEndPoint`: Android AndIodine) plus wildcard DNS traffic that's benign but structurally tunnel-like (many unique subdomains under one domain). This mirrors the dataset paper's own evaluation design: after training, score the model on `heldout.csv` grouped by `source_group` to see (a) recall on tunneling tools it's never seen, and (b) false-positive rate on legitimate-but-unusual traffic. That's a natural next cell to add once a model is trained.

**Binary vs. multi-class.** Every row above keeps `Label` as `benign`/`tunnel`. The tool name (e.g. `dnscat2-cname`, `iodine-txt`) is recoverable from `pcap_file` if you want a multi-class-by-tool-family version later — `TRAIN_FILES` already documents which tool produced each file.

## Part 2 — Model training (BiLSTM + Multi-Head Attention)

Everything below this point is the original modeling pipeline, now pointed at the DNS features generated above (`dataset/train.csv` / `val.csv` / `test.csv`) instead of the original NSL-KDD-style CSVs. It was written for a generic multi-class intrusion dataset; the two changes needed for two-class (`benign`/`tunnel`) DNS data are the output-layer size in `Build_model` and the confusion-matrix shape, both now inferred from the data instead of hardcoded to `6`.

A few cells further down (flagged inline) look like leftovers from earlier experimentation — they reference variables and a `Build_model` signature that don't match what's defined in this notebook, so they'll error if run. They're out of scope for this pass; worth cleaning up or removing separately.

In [ ]:
# Imported via tensorflow.keras instead of bare `keras`. The standalone
# `keras` pip package (Keras 3) has to version-match whatever TensorFlow
# is installed; if the two drift apart (e.g. an old conda TensorFlow next
# to a freshly pip-installed keras) names like `keras.optimizers.Adam` can
# go missing even though `keras.layers` still works. Going through
# tensorflow.keras sidesteps that -- it's guaranteed consistent with
# whatever TensorFlow version is actually installed.
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import KFold
from tensorflow.keras.layers import Dropout, BatchNormalization, Bidirectional, LSTM, MultiHeadAttention, Dense, Input, Concatenate, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

print("tensorflow", tf.__version__, "| keras", keras.__version__)

In [ ]:
seed = 0
tf.random.set_seed(seed)

In [ ]:
# Loaded from the DNS feature-extraction section above (Part 1), which
# writes these three files with a matching schema (features + 'Label').
train = pd.read_csv('dataset/train.csv')
val = pd.read_csv('dataset/val.csv')
test  = pd.read_csv('dataset/test.csv')

In [ ]:
X_train = train.drop('Label', axis=1)
y_train = train['Label']
X_test = test.drop('Label', axis=1)
y_test = test['Label']
X_val = val.drop('Label', axis=1)
y_val = val['Label']

In [ ]:
# get_dummies is applied per-split, but val/test can contain categories
# (e.g. a qtype_name like 'PTR') that don't appear in train, and vice
# versa -- reindexing to X_train_numeric's columns keeps every split's
# feature matrix aligned to what the scaler/model were fit on, filling
# any category missing from that split with 0.
X_train_numeric = pd.get_dummies(X_train, drop_first=True)
X_val_numeric = pd.get_dummies(X_val, drop_first=True).reindex(columns=X_train_numeric.columns, fill_value=0)
X_test_numeric = pd.get_dummies(X_test, drop_first=True).reindex(columns=X_train_numeric.columns, fill_value=0)

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_numeric), columns=X_train_numeric.columns)
X_val_scaled = pd.DataFrame(scaler.transform(X_val_numeric), columns=X_val_numeric.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_numeric), columns=X_test_numeric.columns)

In [ ]:
X_train_scaled_tensor = X_train_scaled.values.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_val_scaled_tensor = X_val_scaled.values.reshape((X_val_scaled.shape[0], 1, X_val_scaled.shape[1]))
X_test_scaled_tensor = X_test_scaled.values.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

In [ ]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

In [ ]:
def Build_model(X_train, y_train, X_val, y_val, X_test, y_test, optimizer,
                n_of_hidden_layers, n_neurons, activation='tanh', dropout_rate=0.3, l2_reg=1e-4,
                epochs=50, batch_size=128, early_stopping=True, verbose=1):
    inputs = Input(shape=(X_train.shape[1], X_train.shape[2]))
    x = inputs
    
    # Adding Bidirectional LSTM layers with Dropout and BatchNormalization.
    # kernel/recurrent L2 added: with only 1 timestep of input, this stack was
    # memorizing the training set (loss -> ~0 within 1 epoch) instead of
    # learning a boundary that holds up on unseen sessions -- L2 plus the
    # smaller n_neurons/n_of_hidden_layers passed in below are what actually
    # rein that in.
    for _ in range(n_of_hidden_layers):
        x = Bidirectional(LSTM(n_neurons, activation=activation, return_sequences=True,
                                kernel_regularizer=l2(l2_reg), recurrent_regularizer=l2(l2_reg)))(x)
        x = Dropout(dropout_rate)(x)
        x = BatchNormalization()(x)
    
    attention = MultiHeadAttention(num_heads=4, key_dim=n_neurons)(x, x)
    x = Concatenate()([x, attention])
    
    x = Flatten()(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(32, activation='relu', kernel_regularizer=l2(l2_reg))(x)
    
    # Inferred from the data instead of hardcoded: the original 6-class
    # NSL-KDD-style labels became binary benign/tunnel DNS labels.
    n_classes = len(np.unique(y_train))
    outputs = Dense(n_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)

    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    es = EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="auto",
        baseline=None,
        restore_best_weights=True,
    )

    # ReduceLROnPlateau: about 1 in 5 runs of this architecture was collapsing
    # to always predicting the majority class and getting stuck there for
    # every remaining epoch (val_loss flat/rising, val_accuracy pinned at the
    # majority-class rate) -- giving a stuck run a shrinking learning rate
    # instead of only a fixed one gives it a chance to escape that plateau
    # rather than early-stopping straight out of it.
    rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=0)

    # class_weight: train is ~1.22:1 benign:tunnel and val ~2.6:1 now that
    # wildcard hard negatives are in both -- balancing the loss keeps that
    # mild imbalance from being what tips an unlucky init toward the
    # always-predict-benign shortcut above.
    classes = np.unique(y_train)
    class_weight = dict(zip(classes, compute_class_weight('balanced', classes=classes, y=y_train)))

    if early_stopping:
        history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=batch_size, callbacks=[es, rlrop], class_weight=class_weight, verbose=verbose)
    else:
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, batch_size=batch_size, callbacks=[rlrop], class_weight=class_weight, verbose=verbose)
    
    train_evaluation = model.evaluate(X_train, y_train, verbose=0)
    test_evaluation = model.evaluate(X_test, y_test, verbose=0)
    validation_evaluation = model.evaluate(X_val, y_val, verbose=0)
    
    return model, history, train_evaluation, validation_evaluation, test_evaluation

In [ ]:
def Build_experiment(X_train, y_train, X_val, y_val, X_test, y_test,
                     n_of_hidden_layers, n_neurons, activation='tanh', dropout_rate=0.3, l2_reg=1e-4, epochs=50, batch_size=128, n_of_models=5, early_stopping=True, verbose=1):
    models_dict = {'models': [], 'history': []}

    models_train_acc = []
    models_test_acc = []
    models_valid_acc = []

    for j in range(n_of_models):
        # Recreate optimizer for each model
        optimizer = Adam(learning_rate=0.001)
        
        # Build model
        model, history, train_evaluation, valid_evaluation, test_evaluation = Build_model(
            X_train, y_train, X_val, y_val, X_test, y_test, optimizer,
            n_of_hidden_layers, n_neurons, activation=activation, dropout_rate=dropout_rate, l2_reg=l2_reg, epochs=epochs, batch_size=batch_size, early_stopping=early_stopping, verbose=verbose
        )
        
        # Save data
        models_train_acc.append(train_evaluation[1])
        models_test_acc.append(test_evaluation[1])
        models_valid_acc.append(valid_evaluation[1])

        models_dict['models'].append(model)
        models_dict['history'].append(history)

    accuracies_dict = {
        "Min_train_acc": min(models_train_acc),
        "Max_train_acc": max(models_train_acc),
        "AVG_train_acc": np.mean(models_train_acc),
        "Min_test_acc": min(models_test_acc),
        "Max_test_acc": max(models_test_acc),
        "AVG_test_acc": np.mean(models_test_acc),
        "Min_valid_acc": min(models_valid_acc),
        "Max_valid_acc": max(models_valid_acc),
        "AVG_valid_acc": np.mean(models_valid_acc)
    }

    return accuracies_dict, models_dict, models_train_acc, models_test_acc, models_valid_acc

In [ ]:
# Perform cross-validation
def cross_validate(X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold = 1
    all_train_acc = []
    all_val_acc = []

    for train_index, val_index in kf.split(X):
        print(f"Training on fold {fold}...")
        X_train_fold, X_val_fold = X[train_index], X[val_index]
        y_train_fold, y_val_fold = y[train_index], y[val_index]
        
        _, model_dict, train_acc, val_acc, _ = Build_experiment(
            X_train_fold, y_train_fold, X_val_fold, y_val_fold, X_val_fold, y_val_fold,
            n_of_hidden_layers=2, n_neurons=50, activation='tanh', dropout_rate=0.3, epochs=50, batch_size=128, n_of_models=1, early_stopping=True, verbose=1
        )
        
        all_train_acc.append(train_acc[0])  # Each fold has one model with one accuracy value
        all_val_acc.append(val_acc[0])
        fold += 1

    return all_train_acc, all_val_acc

# Convert tensors back to arrays for cross-validation
X_combined = np.vstack([X_train_scaled_tensor, X_val_scaled_tensor, X_test_scaled_tensor])
y_combined = np.hstack([y_train_encoded, y_val_encoded, y_test_encoded])

# Perform cross-validation
train_acc, val_acc = cross_validate(X_combined, y_combined)

### Missing piece: nothing above actually built `lstm_models_dict` / `lstm_accuracies_dict`

`cross_validate` (above) calls `Build_experiment` internally but only keeps the per-fold accuracy numbers (`all_train_acc`, `all_val_acc`) — it discards the trained models and full accuracy breakdown. `plot_history`, the confusion matrix, and the accuracy printout below all expect `lstm_models_dict` / `lstm_accuracies_dict`, which only a **direct** call to `Build_experiment` produces. The cell below adds that call.

**Also worth noting**: your folds above converged to ~100% accuracy within a handful of epochs — worth being skeptical of, but not for a data-leakage reason (the model never sees `qname` or the domain string itself; `FEATURE_COLUMNS` deliberately excludes identifiers). The more likely explanation is that this small, curated subset genuinely is easy: these tunneling tools run unthrottled and produce traffic that's wildly different from ordinary browsing (query-name length ~143 chars vs ~14, response TTLs of ~34s vs ~3600s) — that's a known property/critique of DNS-tunnel benchmark datasets like this one, since a "low and slow" evasive tunnel in the real world would look much closer to normal traffic and be far harder to catch. Two things worth doing before trusting this number: (1) score the trained model on `dataset/heldout.csv` (tools/domains the model never trained on) to see if accuracy holds up on genuinely unseen traffic, and (2) treat near-100% on 4-6 training files as "this pipeline works end-to-end," not yet as "this generalizes" — that needs the full 131-file dataset and more tool diversity per the scaling-up notes earlier in this notebook.

In [ ]:
# n_of_models=5 trains 5 independent models so Build_experiment can report a
# min/max/avg spread (that's what plot_history's 2x3 grid of subplots is
# for). Based on your cross_validate run above (~12s/epoch, converging in
# 10-15 epochs), this should take on the order of 10-15 minutes total. Drop
# n_of_models to 1 or 2 first if you just want to confirm the rest of the
# pipeline (plot_history / confusion matrix / accuracy printout) runs.
# n_of_hidden_layers/n_neurons cut and dropout_rate/l2_reg raised relative to
# the original run: with wildcard hard negatives now in train/val (see Part 1
# above), the original capacity (2 layers x 50 units, dropout 0.3, no L2)
# memorized the training set within 1 epoch and collapsed on validation --
# always predicting "benign" (val/test accuracy pinned at the majority-class
# baseline). This smaller, more regularized config is what actually forces
# the model to learn a boundary that holds up on unseen sessions.
lstm_accuracies_dict, lstm_models_dict, lstm_train_acc, lstm_test_acc, lstm_valid_acc = Build_experiment(
    X_train_scaled_tensor, y_train_encoded, X_val_scaled_tensor, y_val_encoded, X_test_scaled_tensor, y_test_encoded,
    n_of_hidden_layers=1, n_neurons=24, activation='tanh', dropout_rate=0.5, l2_reg=1e-4,
    epochs=50, batch_size=128, n_of_models=5, early_stopping=True, verbose=1
)

In [ ]:
def plot_history(dict_of_lists, type='loss'):
    axis_idx = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))

    for i in range(len(dict_of_lists)):
        ax = axes[axis_idx[i][0]][axis_idx[i][1]]
        # Summarize history for accuracy
        ax.plot(dict_of_lists[i].history[type])
        ax.plot(dict_of_lists[i].history[f'val_{type}'])
        ax.set_title(f'Model {i+1} {type.title()}', size=15)
        ax.set_ylabel(f'{type.title()}', size=10)
        ax.set_xlabel('Epoch', size=10)
        ax.legend(['train', 'test'], loc='upper left')
    plt.suptitle(f'Models {type.title()} per Epoch', size=15, y=0.93)
    plt.show()

plot_history(lstm_models_dict['history'], type='accuracy')
plot_history(lstm_models_dict['history'], type='loss')

In [ ]:
# Display the average confusion matrix
n_classes = len(label_encoder.classes_)  # was hardcoded to 6 for the old multi-class labels
cm = np.zeros(shape=(n_classes, n_classes))
for model in lstm_models_dict['models']:
    pred = model.predict(X_test_scaled_tensor).argmax(axis=1)
    cm += confusion_matrix(y_test_encoded, pred)
cm_avg = cm / len(lstm_models_dict['models'])

# Set custom color palette
colors = ["#FF0B04", "#4374B3"]
sns.set_palette(sns.color_palette(colors))
sns.set(font_scale=1.5)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_avg, display_labels=label_encoder.classes_)
disp = disp.plot(cmap=plt.cm.Blues, values_format='g')

fig = disp.ax_.get_figure()
fig.set_figwidth(10)
fig.set_figheight(10)

plt.title('Confusion Matrix of the Highest Accuracy')
plt.grid(False)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(train_acc) + 1), train_acc, marker='o', label='Train Accuracy')
plt.plot(range(1, len(val_acc) + 1), val_acc, marker='x', label='Validation Accuracy')

plt.title('Cross-Validation Accuracy Scores for All Folds')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# Min/Max alongside Average: the average alone hides a real failure mode
# here -- a collapsed single run (always predicting the majority class) pulls
# the average down while looking, at a glance, like "slightly lower accuracy"
# rather than what it actually is (4 good runs + 1 broken one). Min close to
# Max is the signal that all n_of_models runs actually converged properly.
for split in ["train", "valid", "test"]:
    print(f"{split.title()} Accuracy -- "
          f"min={lstm_accuracies_dict.get(f'Min_{split}_acc', float('nan')):.4f}  "
          f"max={lstm_accuracies_dict.get(f'Max_{split}_acc', float('nan')):.4f}  "
          f"avg={lstm_accuracies_dict.get(f'AVG_{split}_acc', float('nan')):.4f}")


## Real generalization test: scoring against `dataset/heldout.csv`

Everything above (train/val/test) came from the same small, curated pool of files, so ~100% accuracy there mostly proves the pipeline runs end-to-end -- not that the model generalizes. `heldout.csv` was never touched during training, model selection, or scaling/encoding fit, and it's built specifically to be harder:

- **`unkownTunnel` / `crossEndPoint`** -- tunnel tools the model has never seen (cobaltstrike, ozymandns, a different Iodine endpoint). Tests **false negatives**: does it catch genuinely novel tunnels, or did it just memorize the training tools' fingerprints?
- **`wildcard`** -- legitimate DNS traffic that is structurally tunnel-*like* (many subdomains, high query rate). Tests **false positives**: does the model actually learn tunneling behavior, or does it just flag "unusual DNS" as malicious?

Run the cell below after the training cell above has finished (it reuses the already-fit `scaler`, `label_encoder`, and the 5 trained models in `lstm_models_dict`).

In [ ]:
# heldout.csv was NEVER used for training/validation/model-selection --
# this is the actual test of whether the model generalizes, not just
# whether it memorized this small train/val/test pool.
heldout = pd.read_csv('dataset/heldout.csv')

X_heldout = heldout.drop(['Label', 'source_group'], axis=1)
y_heldout = heldout['Label']
groups_heldout = heldout['source_group']

# Same alignment trick as train/val/test: reindex to the columns the
# scaler/model were actually fit on.
X_heldout_numeric = pd.get_dummies(X_heldout, drop_first=True).reindex(
    columns=X_train_numeric.columns, fill_value=0
)
X_heldout_scaled = pd.DataFrame(
    scaler.transform(X_heldout_numeric), columns=X_heldout_numeric.columns
)
X_heldout_scaled_tensor = X_heldout_scaled.values.reshape(
    (X_heldout_scaled.shape[0], 1, X_heldout_scaled.shape[1])
)
y_heldout_encoded = label_encoder.transform(y_heldout)

heldout_accs = []
for i, m in enumerate(lstm_models_dict['models']):
    loss, acc = m.evaluate(X_heldout_scaled_tensor, y_heldout_encoded, verbose=0)
    heldout_accs.append(acc)
    print(f"model {i}: heldout accuracy = {acc:.4f}")

print(f"\nheldout accuracy -- min={min(heldout_accs):.4f}  "
      f"max={max(heldout_accs):.4f}  avg={np.mean(heldout_accs):.4f}")
print(f"(compare to test accuracy avg={lstm_accuracies_dict['AVG_test_acc']:.4f} from the in-distribution split above)")

# Per-group breakdown with the best-performing model on heldout, so we can
# see *which* kind of traffic (if any) actually trips the model up.
best_model = lstm_models_dict['models'][int(np.argmax(heldout_accs))]
y_pred_heldout = np.argmax(best_model.predict(X_heldout_scaled_tensor, verbose=0), axis=1)

results = pd.DataFrame({
    'group': groups_heldout,
    'true': y_heldout,
    'pred': label_encoder.inverse_transform(y_pred_heldout),
})
results['correct'] = results['true'] == results['pred']

print("\nPer-group accuracy (best model on heldout):")
print(results.groupby('group')['correct'].agg(['mean', 'count']).rename(
    columns={'mean': 'accuracy', 'count': 'n_rows'}
))

cm_heldout = confusion_matrix(y_heldout_encoded, y_pred_heldout)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_heldout, display_labels=label_encoder.classes_)
disp.plot(cmap='Blues')
plt.title('Confusion matrix -- held-out (unseen tools + wildcard benign)')
plt.show()

### Removed: duplicate second cross-validation block

This cell called `Build_model(input_shape=(...))`, but the real `Build_model` defined above takes `(X_train, y_train, X_val, y_val, X_test, y_test, optimizer, ...)` and trains internally -- there's no `input_shape`-only factory version, so this always raised `TypeError`. It duplicated the `cross_validate` cell above (same 5-fold CV over `X_train_scaled_tensor`), which already ran successfully, so it and its plot cell were removed rather than rewritten.

In [ ]:
!jupyter nbconvert "NetworkIntrusionDetection_1_1_1.ipynb" --to pdf

In [ ]:
!xelatex --version